# Voice on Google Colab

- **GPU:** Runtime -> Change runtime type -> **T4** (recommended).
- **Order:** cell 1 (install) -> cell 2 (backend) -> cell 3 (Gradio). Same as local: `make run_all` then `make run_ui`.
- **`GRADIO_SHARE=true`:** public share link; omit it to use the UI only inside the Colab tab.

In [ ]:
# First run: fish-speech clone + HF weights take a while. No .env yet -> !cp .env.example .env and set ANYTHINGLLM_* (required for LLM / Voice Chat).
# 1. Clone repo + system deps
!apt-get update -qq && apt-get install -y -qq ffmpeg
![ -d voice-cloning ] || git clone https://github.com/singswap/voice-cloning.git
%cd voice-cloning

# 2. requirements
!pip install -q -r requirements.txt
!pip install -q huggingface_hub

# 3. fish-speech + numpy pin
![ -d /root/fish-speech ] || git clone --depth 1 https://github.com/fishaudio/fish-speech.git /root/fish-speech
!pip install -q -e /root/fish-speech
!pip install -q "numpy==1.26.4"

# 4. weights (download only if missing)
from pathlib import Path
from huggingface_hub import snapshot_download

_ckpt = Path("models/fish-speech-1.5/firefly-gan-vq-fsq-8x1024-21hz-generator.pth")
if not _ckpt.is_file():
    snapshot_download(repo_id="fishaudio/fish-speech-1.5", local_dir="models/fish-speech-1.5")

# Optional: pyngrok for extra tunnels
# !pip install pyngrok

In [ ]:
# Same ports as Makefile. Check gateway: curl localhost:8000/health
import os
import time

os.makedirs("logs", exist_ok=True)

print("Starting backend...")
!nohup python -m uvicorn services.stt.app:app --host 0.0.0.0 --port 8001 > logs/stt.log 2>&1 &
!TTS_ENGINES=fish nohup python -m uvicorn services.tts.app:app --host 0.0.0.0 --port 8002 > logs/tts.log 2>&1 &
!nohup python -m uvicorn services.rvc.app:app --host 0.0.0.0 --port 8003 > logs/rvc.log 2>&1 &
!nohup python -m uvicorn services.llm.app:app --host 0.0.0.0 --port 8004 > logs/llm.log 2>&1 &
!nohup python -m uvicorn gateway.app:app --host 0.0.0.0 --port 8000 > logs/gateway.log 2>&1 &

print("Waiting 15s for services to start...")
time.sleep(15)
!curl http://localhost:8000/health

In [ ]:
# Gradio UI. Stop runtime when done to free the GPU.
!GRADIO_SHARE=true python -m ui.app